# Operational Data Sourcing Lab

This notebook demonstrates how to execute the Week 4 lab using the provided resources:

- `operational_events_5000.csv`: 5,000 operational records
- `mock_weather_api_response.json`: API-style JSON response
- `mock_market_prices.html`: scrapeable HTML table
- `ops_lab.db`: SQLite operational database
- `data_dictionary.csv`: field descriptions

The lab illustrates API connection concepts, web scraping, SQL querying, and a simple ETL flow for an operational environment.

## 1. Setup

Run this notebook from the same folder as the lab files. The cells below use relative file paths so the notebook remains portable.

In [1]:
from pathlib import Path
import json
import sqlite3

import pandas as pd

LAB_DIR = Path.cwd()
CSV_PATH = LAB_DIR / "operational_events_5000.csv"
API_PATH = LAB_DIR / "mock_weather_api_response.json"
HTML_PATH = LAB_DIR / "mock_market_prices.html"
DB_PATH = LAB_DIR / "ops_lab.db"
DICTIONARY_PATH = LAB_DIR / "data_dictionary.csv"

for path in [CSV_PATH, API_PATH, HTML_PATH, DB_PATH, DICTIONARY_PATH]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

operational_events_5000.csv: FOUND
mock_weather_api_response.json: FOUND
mock_market_prices.html: FOUND
ops_lab.db: FOUND
data_dictionary.csv: FOUND


## 2. Review the data dictionary

Before analyzing a dataset, inspect the meaning of important columns.

In [2]:
data_dictionary = pd.read_csv(DICTIONARY_PATH)
data_dictionary

,column_name,description
0,record_id,Unique operational event identifier.
1,event_timestamp_utc,Timestamp when the operational reading occurred.
2,source_system,Origin type used to practice multi-source extr...
3,flow_rate_lpm,Flow rate in litres per minute; blank for miss...
4,pressure_psi,Pipeline pressure in PSI; blank for missing-re...
5,api_weather_condition,Weather field from the API-style source; null ...
6,scraped_market_price_usd_bbl,Market price field from the scrapeable HTML so...
7,quality_issue_type,"VALID, MISSING_RECORD, or NULL_RECORD."
8,missing_record_flag,1 for the 800 records with missing operational...
9,null_record_flag,1 for the 200 records with null API/scraped pa...


## 3. Extract from a CSV source

This is the main operational dataset. It simulates readings from depots, pipeline segments, APIs, scraped market data, and database systems.

In [6]:
ops_df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(ops_df):,}")
print(f"Columns: {ops_df.shape[1]}")
ops_df.head(8)

Rows: 5,000
Columns: 27


,record_id,event_timestamp_utc,source_system,depot_id,depot_name,region,product_code,tank_id,pipeline_segment,flow_rate_lpm,...,scraped_market_price_usd_bbl,operator_shift,incident_flag,maintenance_required,ingested_at_utc,batch_id,quality_issue_type,missing_record_flag,null_record_flag,source_url
0,OPS-00001,2026-03-01T00:00:00Z,SQL_DB,DPT001,Nairobi Terminal,Central,IK,T03,Eldoret-Kisumu,NaN,...,91.40,Day,0,0,2026-03-01T00:15:00Z,BATCH-20260301,MISSING_RECORD,1,0,https://ops.example.local/dpt001/ik
1,OPS-00002,2026-03-01T00:15:00Z,SCADA,DPT005,Nakuru Depot,Rift Valley,PMS,T10,Nairobi-Eldoret,616.90,...,78.09,Day,0,0,2026-03-01T00:27:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt005/pms
2,OPS-00003,2026-03-01T00:30:00Z,WEB_SCRAPE,DPT002,Mombasa Terminal,Coast,JET-A1,T17,Eldoret-Kisumu,1222.52,...,98.80,Day,0,0,2026-03-01T00:35:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt002/jet-a1
3,OPS-00004,2026-03-01T00:45:00Z,SCADA,DPT001,Nairobi Terminal,Central,IK,T18,Nairobi-Eldoret,508.40,...,74.76,Day,0,0,2026-03-01T01:11:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt001/ik
4,OPS-00005,2026-03-01T01:00:00Z,WEB_SCRAPE,DPT005,Nakuru Depot,Rift Valley,IK,T05,Mombasa-Nairobi,535.83,...,93.49,Day,0,0,2026-03-01T01:34:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt005/ik
5,OPS-00006,2026-03-01T01:15:00Z,SQL_DB,DPT005,Nakuru Depot,Rift Valley,IK,T14,Nairobi-Eldoret,994.28,...,85.11,Day,0,0,2026-03-01T01:24:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt005/ik
6,OPS-00007,2026-03-01T01:30:00Z,WEB_SCRAPE,DPT006,Morendat Training Hub,Central,PMS,T01,Mombasa-Nairobi,752.42,...,101.01,Day,0,1,2026-03-01T01:58:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt006/pms
7,OPS-00008,2026-03-01T01:45:00Z,SCADA,DPT002,Mombasa Terminal,Coast,PMS,T15,Nairobi-Eldoret,857.87,...,88.83,Day,0,0,2026-03-01T01:53:00Z,BATCH-20260301,VALID,0,0,https://ops.example.local/dpt002/pms


## 4. Validate required lab counts

The lab dataset was designed with exact data-quality targets:

- 5,000 total rows
- 800 missing-record rows
- 200 null-record rows

In [7]:
quality_counts = {
    "total_rows": len(ops_df),
    "missing_record_rows": int(ops_df["missing_record_flag"].sum()),
    "null_record_rows": int(ops_df["null_record_flag"].sum()),
    "valid_rows": int(((ops_df["missing_record_flag"] == 0) & (ops_df["null_record_flag"] == 0)).sum()),
}

quality_counts

{'total_rows': 5000,
 'missing_record_rows': 800,
 'null_record_rows': 200,
 'valid_rows': 4000}

In [9]:
assert quality_counts["total_rows"] == 5000
assert quality_counts["missing_record_rows"] == 800
assert quality_counts["null_record_rows"] == 200

print("Quality-count validation passed.")

Quality-count validation passed.


## 5. API connection concept

The JSON file is a local stand-in for an API response. In a live API workflow, the same logic would follow a `requests.get(...)` call.

In [11]:
with API_PATH.open("r", encoding="utf-8") as handle:
    api_payload = json.load(handle)

weather_df = pd.DataFrame(api_payload["records"])
print(f"API-style records: {len(weather_df):,}")
weather_df.head(3)

API-style records: 60


,depot_id,depot_name,region,observed_at_utc,weather_condition,outside_temperature_c,humidity_pct,wind_speed_kph,source
0,DPT001,Nairobi Terminal,Central,2026-03-31T06:00:00Z,Windy,26.3,56,2.1,mock-weather-api
1,DPT002,Mombasa Terminal,Coast,2026-03-31T06:00:00Z,Cloudy,20.1,78,27.8,mock-weather-api
2,DPT003,Eldoret Depot,Rift Valley,2026-03-31T06:00:00Z,Heavy Rain,22.9,83,11.9,mock-weather-api


In [13]:
# Live API pattern for reference only:
#
# import requests
# response = requests.get(API_URL, headers={"Authorization": f"Bearer {API_KEY}"}, timeout=30)
# response.raise_for_status()
# live_api_df = pd.DataFrame(response.json()["records"])

weather_summary = (
    weather_df.groupby(["region", "weather_condition"], as_index=False)
    .agg(avg_temperature_c=("outside_temperature_c", "mean"), avg_humidity_pct=("humidity_pct", "mean"))
    .round(2)
)

weather_summary.head(20)

,region,weather_condition,avg_temperature_c,avg_humidity_pct
0,Central,Clear,22.30,62.50
1,Central,Cloudy,24.65,64.50
2,Central,Heavy Rain,27.35,59.25
3,Central,Light Rain,24.74,65.40
4,Central,Windy,22.38,66.80
5,Coast,Clear,21.23,60.00
6,Coast,Cloudy,20.10,71.50
7,Coast,Heavy Rain,23.45,76.00
8,Coast,Light Rain,32.40,58.00
9,Coast,Windy,24.30,54.50


## 6. Web scraping concept

The HTML file contains a normal table. `pandas.read_html` parses tables from web pages or local HTML files.

In [14]:
market_tables = pd.read_html(HTML_PATH)
market_df = market_tables[0]

print(f"Tables found: {len(market_tables)}")
print(f"Market price rows: {len(market_df):,}")
market_df.head()

Tables found: 1
Market price rows: 31


,trade_date,brent_usd_bbl,usd_kes_rate,supply_status
0,2026-03-01,98.92,1.276,Normal
1,2026-03-02,88.86,1.189,Tight
2,2026-03-03,99.06,1.083,Tight
3,2026-03-04,80.46,1.224,Delayed
4,2026-03-05,100.71,1.278,Delayed


In [15]:
market_df["trade_date"] = pd.to_datetime(market_df["trade_date"])

market_df.describe(include="all")

,trade_date,brent_usd_bbl,usd_kes_rate,supply_status
count,31,31.000000,31.000000,31
unique,NaN,NaN,NaN,3
top,NaN,NaN,NaN,Normal
freq,NaN,NaN,NaN,14
mean,2026-03-16 00:00:00,93.458387,1.209258,NaN
min,2026-03-01 00:00:00,74.030000,1.083000,NaN
25%,2026-03-08 12:00:00,87.630000,1.165000,NaN
50%,2026-03-16 00:00:00,95.830000,1.224000,NaN
75%,2026-03-23 12:00:00,100.420000,1.270500,NaN
max,2026-03-31 00:00:00,110.910000,1.307000,NaN


## 7. SQL querying concept

The SQLite database contains the same operational events plus a small depot reference table. This mirrors an operational database used by dashboards and reports.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;",
        conn,
    )

tables

In [ ]:
query = """
SELECT
    depot_name,
    product_code,
    COUNT(*) AS total_records,
    SUM(missing_record_flag) AS missing_records,
    SUM(null_record_flag) AS null_records,
    ROUND(AVG(pressure_psi), 2) AS avg_pressure_psi
FROM operational_events
GROUP BY depot_name, product_code
ORDER BY missing_records DESC
LIMIT 10;
"""

with sqlite3.connect(DB_PATH) as conn:
    sql_summary = pd.read_sql_query(query, conn)

sql_summary

## 8. Transform and clean

This step converts timestamps, creates a clean subset, and prepares records for loading. In production, each operation would usually live inside a reusable function.

In [ ]:
transformed_df = ops_df.copy()

transformed_df["event_timestamp_utc"] = pd.to_datetime(transformed_df["event_timestamp_utc"], utc=True)
transformed_df["ingested_at_utc"] = pd.to_datetime(transformed_df["ingested_at_utc"], utc=True)

numeric_columns = [
    "flow_rate_lpm",
    "pressure_psi",
    "temperature_c",
    "inventory_litres",
    "api_temperature_c",
    "scraped_market_price_usd_bbl",
]

for column in numeric_columns:
    transformed_df[column] = pd.to_numeric(transformed_df[column], errors="coerce")

clean_df = transformed_df.query("missing_record_flag == 0 and null_record_flag == 0").copy()

print(f"Clean rows ready for loading: {len(clean_df):,}")
clean_df.head()

## 9. Data-quality checks

These checks imitate the kind of rules you would formalize in Great Expectations or another validation framework.

In [ ]:
checks = {
    "record_id_unique": clean_df["record_id"].is_unique,
    "pressure_not_null": clean_df["pressure_psi"].notna().all(),
    "pressure_between_0_and_200": clean_df["pressure_psi"].between(0, 200).all(),
    "flow_rate_positive": (clean_df["flow_rate_lpm"] > 0).all(),
    "accepted_pump_status": clean_df["pump_status"].isin(["RUNNING", "IDLE", "STOPPED", "ALARM"]).all(),
    "minimum_clean_row_count": len(clean_df) >= 100,
}

pd.DataFrame(
    [{"check": check_name, "passed": passed} for check_name, passed in checks.items()]
)

In [ ]:
if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError(f"Data-quality checks failed: {failed}")

print("All data-quality checks passed. Proceeding to load phase.")

## 10. Load clean data idempotently

The load below writes a clean table back into SQLite. Using `if_exists='replace'` makes this demonstration idempotent: re-running the notebook replaces the clean output table instead of appending duplicates.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    clean_df.to_sql("clean_operational_events", conn, if_exists="replace", index=False)

with sqlite3.connect(DB_PATH) as conn:
    loaded_count = conn.execute("SELECT COUNT(*) FROM clean_operational_events;").fetchone()[0]

print(f"Loaded rows in clean_operational_events: {loaded_count:,}")

## 11. Final operational summary

This summary is the kind of output a pipeline could send to a daily report or dashboard.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    final_summary = pd.read_sql_query(
        """
        SELECT
            depot_name,
            COUNT(*) AS clean_records,
            ROUND(AVG(flow_rate_lpm), 2) AS avg_flow_rate_lpm,
            ROUND(AVG(pressure_psi), 2) AS avg_pressure_psi,
            SUM(incident_flag) AS incidents,
            SUM(maintenance_required) AS maintenance_required
        FROM clean_operational_events
        GROUP BY depot_name
        ORDER BY incidents DESC, clean_records DESC;
        """,
        conn,
    )

final_summary

## 12. Suggested lab extension

To extend this notebook into a production-style Week 4 pipeline:

1. Move extraction, transformation, validation, and loading into separate Python functions.
2. Add `logging` messages at the start and end of every phase.
3. Store paths and database settings in a `.env` file.
4. Replace the manual checks with a Great Expectations validation suite.
5. Schedule the script using cron or Windows Task Scheduler.